# SoterLLM v7 — Colab GPU Training

**Before anything: Runtime -> Change runtime type -> T4 GPU -> Save.**
Then check the top right says **Connected** with a green tick.

1. Run the cells top to bottom. Cell 2 shows a **Choose Files** button — pick
   `soterai-train-bundle.zip` from `C:\Users\USER\OneDrive\Desktop\Ai-Agent-Security-Guard`.
   (No need for the Files sidebar; that only offers upload once a runtime is connected,
   and the 12.4 MB upload must finish before the cell continues.)
2. Download `ml-classifier-v7.zip` from the last cell.

Do **not** run other cells while training — they fight for GPU memory.
Expect roughly 1.5-3 hours on a T4 for 4 epochs over 109,054 rows.

> **If you edited this notebook locally, re-upload it.** An open Colab tab keeps
> serving the version it loaded; File -> Upload notebook (or Ctrl+O) to replace it.
> A stale tab is why an already-fixed cell can keep throwing the same error.

In [ ]:
# 1/5  Confirm the GPU is really attached before spending an hour finding out it isn't.
import subprocess, torch

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi produced nothing")
assert torch.cuda.is_available(), (
    "No CUDA. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Factory reset runtime, then re-run."
)
print("torch", torch.__version__, "| device:", torch.cuda.get_device_name(0))

In [ ]:
# 2/5  Get the bundle into the runtime, then unpack it.
#
# Three sources, tried in order, so this does not depend on the Files sidebar
# (that sidebar only offers an upload button once a runtime is connected, which
# is why "there is no upload option" happens on a fresh/disconnected tab):
#   1. already in the working directory  2. Google Drive  3. inline upload button
#
# Unpacked with Python zipfile rather than `unzip -q` because a bundle zipped on
# Windows with Compress-Archive stores backslash separators, which extract as ONE
# file literally named "scripts\ml\train-soterllm-v4.py" and make cell 3 report
# the trainer "missing from bundle".
import hashlib, os, zipfile

BUNDLE = "soterai-train-bundle.zip"
EXPECTED_SHA = "c5a4184abe440d59121a079f2af0e9b8cd4423797b63669d3cec3af331695e6e"

if not os.path.exists(BUNDLE):
    for cand in (
        "/content/drive/MyDrive/soterai-train-bundle.zip",
        "/content/drive/MyDrive/soterai/soterai-train-bundle.zip",
    ):
        if os.path.exists(cand):
            print(f"found in Drive: {cand}")
            os.symlink(cand, BUNDLE)
            break

if not os.path.exists(BUNDLE):
    print(f"{BUNDLE} not found in {os.getcwd()}.")
    print("Pick the file from your computer with the button below")
    print("(project root: C:\\Users\\USER\\OneDrive\\Desktop\\Ai-Agent-Security-Guard).\n")
    print("If no button appears, the runtime is not connected: click Connect")
    print("(top right), wait for the green tick, then re-run this cell.\n")
    from google.colab import files
    for name in files.upload():                     # renders a Choose Files button
        if name != BUNDLE and name.endswith(".zip"):
            os.rename(name, BUNDLE)

if not os.path.exists(BUNDLE):
    raise SystemExit("no bundle in the runtime — nothing was uploaded, re-run this cell")

sha = hashlib.sha256(open(BUNDLE, "rb").read()).hexdigest()
print(f"\n{BUNDLE}  {os.path.getsize(BUNDLE)/1e6:.1f} MB  sha256 {sha[:16]}...")
if sha != EXPECTED_SHA:
    print(
        "\n  WARNING: not the bundle this notebook was built against\n"
        f"  (expected {EXPECTED_SHA[:16]}..., about 12.4 MB).\n"
        "  A stale upload is the usual cause of 'missing from bundle' below.\n"
        "  Rebuild with scripts/ml/colab/_rebuild_bundle.py and re-upload."
    )

with zipfile.ZipFile(BUNDLE) as z:
    bad = z.testzip()
    if bad is not None:
        raise SystemExit(f"corrupt archive (bad member: {bad}) — upload was truncated, re-upload")
    for info in z.infolist():
        if info.is_dir():
            continue
        dest = os.path.join(*info.filename.replace("\\", "/").split("/"))  # Windows-zip repair
        os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
        with z.open(info) as src, open(dest, "wb") as out:
            out.write(src.read())
        print(f"  extracted  {dest:44} {os.path.getsize(dest)/1e6:6.1f} MB")

# onnxscript is REQUIRED here: torch 2.9's exporter dies without it, after
# spending 45+ minutes training. It belongs in cell 2 so a fresh run never
# crashes at the export stage.
!pip -q install transformers onnx onnxruntime onnxscript scikit-learn

In [ ]:
# 3/5  Preflight. The split guard is the point: a random split leaked ~35.8% of val
# via augmentation siblings once and produced a fake 99.29% F1. Parsed with ast
# because the trainer's docstring *mentions* random_split to document that mistake.
import ast, os, zipfile

REQUIRED = (
    "scripts/ml/train-soterllm-v4.py",
    "datasets/ml-augmented-v7.jsonl",
    "datasets/crossdist-eval-v3.jsonl",
)

missing = [p for p in REQUIRED if not os.path.exists(p)]
if missing:
    print("MISSING FROM THE WORKING DIRECTORY:")
    for p in missing:
        print("   ", p)
    print(f"\ncwd = {os.getcwd()}\nwhat is actually here:")
    for name in sorted(os.listdir(".")):
        print("   ", name + ("/" if os.path.isdir(name) else ""))
    if os.path.exists("soterai-train-bundle.zip"):
        with zipfile.ZipFile("soterai-train-bundle.zip") as z:
            print("\nthe uploaded bundle contains:")
            for n in z.namelist():
                print("   ", repr(n))
        print(
            "\n-> Run the cell titled '2/5' first; it extracts the bundle. If the names\n"
            "   above are missing entries or show backslashes, the upload is stale:\n"
            "   rebuild with scripts/ml/colab/_rebuild_bundle.py and re-upload, then\n"
            "   re-run the '2/5' cell."
        )
    else:
        print(
            "\n-> The bundle zip is not here either. Run the cell titled '2/5' — it\n"
            "   shows a Choose Files button to upload it."
        )
    raise SystemExit("preflight failed — see the diagnosis above")

for p in REQUIRED:
    print(f"  OK  {p:44} {os.path.getsize(p)/1e6:6.1f} MB")

src = open("scripts/ml/train-soterllm-v4.py", encoding="utf-8").read()
called = {
    n.func.id if isinstance(n.func, ast.Name) else getattr(n.func, "attr", "")
    for n in ast.walk(ast.parse(src))
    if isinstance(n, ast.Call)
}
assert "group_aware_three_way_split" in called, "trainer never CALLS the group-aware split"
assert "random_split" not in called, "trainer calls random_split — siblings would leak into val"
assert "group_key_for" in src, "trainer lost group_key_for"

rows = sum(1 for _ in open("datasets/ml-augmented-v7.jsonl", encoding="utf-8"))
print(f"\n  leak-free primitives present. corpus rows: {rows}")

In [ ]:
# 4/5  The real training run — full 109K corpus.
# If you hit CUDA out-of-memory, drop --batch-size to 32 and re-run this cell only.
#
# Artifacts are mirrored to Drive as they appear. Colab wipes /content whenever the
# runtime is recycled — reconnecting, uploading a new notebook, an idle timeout —
# and that silently destroys a finished 45-minute train. Drive outlives the runtime.
from google.colab import drive

drive.mount("/content/drive")

import os, shutil, subprocess, threading, time

OUT = "models/ml-classifier-v7"
BACKUP = "/content/drive/MyDrive/soterai-v7-artifacts"
os.makedirs(BACKUP, exist_ok=True)

# Copy with shutil, not rsync: rsync is not guaranteed present on a Colab image,
# and the mirror is the ONLY thing standing between a recycled runtime and another
# lost 45 minutes — it must not be the thing that quietly does nothing.
# Written to a .part file then os.replace()'d, so a mid-copy recycle cannot leave a
# half-written pytorch_model.bin on Drive that later looks restorable.
_stop = threading.Event()
_mirror_log = {"copies": 0, "errors": []}


def _mirror_once():
    if not os.path.isdir(OUT):
        return
    for name in sorted(os.listdir(OUT)):
        src = os.path.join(OUT, name)
        if not os.path.isfile(src):
            continue
        dst = os.path.join(BACKUP, name)
        try:
            if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
                if os.path.getmtime(dst) >= os.path.getmtime(src):
                    continue
            tmp = dst + ".part"
            shutil.copy2(src, tmp)
            os.replace(tmp, dst)
            _mirror_log["copies"] += 1
        except OSError as e:
            _mirror_log["errors"].append(f"{name}: {e}")


def _mirror_loop():
    # Every 2 min: pytorch_model.bin is rewritten at each val-F1 improvement,
    # so the worst case loses one epoch, not the whole run.
    while not _stop.wait(120):
        _mirror_once()


# Prove the path is writable NOW rather than discovering it after training.
_probe = os.path.join(BACKUP, ".write-probe")
with open(_probe, "w") as f:
    f.write("ok")
os.remove(_probe)
print(f"mirroring to: {BACKUP}  (verified writable)")

threading.Thread(target=_mirror_loop, daemon=True).start()

!python scripts/ml/train-soterllm-v4.py \
    --train-datasets datasets/ml-augmented-v7.jsonl \
    --epochs 4 --batch-size 64 --max-length 256 \
    --output-dir models/ml-classifier-v7

_stop.set()
time.sleep(1)
_mirror_once()

print(f"\nfinal mirror to Drive ({_mirror_log['copies']} file copies during the run):")
for n in sorted(os.listdir(BACKUP)):
    p = os.path.join(BACKUP, n)
    if os.path.isfile(p):
        print(f"  {n:28} {os.path.getsize(p)/1e6:7.2f} MB")
if _mirror_log["errors"]:
    print("\n  mirror errors (Drive copy failed — /content is now the only copy):")
    for e in _mirror_log["errors"][-5:]:
        print("   ", e)

print(
    "\nIf 'Export ONNX' crashed but pytorch_model.bin is listed above, run the NEXT\n"
    "cell (the one titled '4b/5 RECOVERY') — do NOT re-train, and do NOT re-upload\n"
    "this notebook, because either one recycles the runtime and wipes /content."
)

In [ ]:
# 4b/5  RECOVERY for a crashed export — re-exports WITHOUT retraining.
#
# Torch 2.9's ONNX exporter needs `onnxscript`. Without it a run can finish
# training, write calibration.json, then die at "Export ONNX" with
# `ModuleNotFoundError: No module named 'onnxscript'`.
#
# The weights are NOT lost to that crash: pytorch_model.bin is saved at the end of
# every epoch that improves val F1. This cell rebuilds the model from that
# checkpoint and re-runs only the export + artifact-writing stages (~2 min).
#
# They ARE lost if the runtime was recycled (reconnect, new notebook upload, idle
# timeout) — /content is wiped. The training cell mirrors to Drive so that is
# survivable; this cell restores from that mirror automatically.
#
# Run this INSTEAD of re-training. Then run the LAST cell ("5/5") to download.
import json, os, shutil, subprocess, sys, time
import torch

OUT = "models/ml-classifier-v7"
BIN = os.path.join(OUT, "pytorch_model.bin")
MIRROR = "/content/drive/MyDrive/soterai-v7-artifacts"

if not os.path.exists(BIN):
    if not os.path.isdir(MIRROR):
        try:
            from google.colab import drive

            drive.mount("/content/drive")
        except Exception as e:
            print("could not mount Drive:", e)
    if os.path.exists(os.path.join(MIRROR, "pytorch_model.bin")):
        print(f"restoring from Drive mirror: {MIRROR}")
        os.makedirs(OUT, exist_ok=True)
        # shutil, not rsync: rsync may not exist on the image, and a restore that
        # silently copies nothing looks exactly like genuinely lost weights.
        for _n in sorted(os.listdir(MIRROR)):
            _s = os.path.join(MIRROR, _n)
            if os.path.isfile(_s) and not _n.endswith(".part"):
                shutil.copy2(_s, os.path.join(OUT, _n))
                print(f"  restored {_n:28} {os.path.getsize(_s)/1e6:7.2f} MB")

assert os.path.exists(BIN), (
    f"no checkpoint at {BIN}, and no Drive mirror to restore from.\n"
    "The runtime was recycled and /content was wiped, so the trained weights are "
    "gone — a full re-train is the only path. Run scripts/ml/colab/"
    "_diagnose_paste_me.py to confirm before spending another 45 GPU-minutes."
)

print("installing onnxscript (needed by torch.onnx.export on torch 2.9)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxscript"], check=True)
import transformers  # noqa: E402

# Load the trainer by file path: "train-soterllm-v4.py" has hyphens, so a plain
# `import` can't name it. This does NOT retrain — spec_from_file_location sets
# __name__ to "soterllm_trainer", so the `if __name__ == "__main__"` tail is dead.
import importlib.util  # noqa: E402

spec = importlib.util.spec_from_file_location(
    "soterllm_trainer", "scripts/ml/train-soterllm-v4.py"
)
T = importlib.util.module_from_spec(spec)
spec.loader.exec_module(T)

# ── Load checkpoint into the exact same architecture (dropout=0.15, default) ──
t0 = time.time()
model = T.SoterLLMv4(num_labels=len(T.ALL_LABELS), dropout=0.15)
model.load_state_dict(torch.load(BIN, map_location="cpu"))
model.eval()
print(f"checkpoint loaded ({time.time() - t0:.1f}s)")

# ── Rebuild tokenizer, then recompute val metrics on the loaded checkpoint ──
# Recomputed rather than transcribed from the crashed run's stdout: a number
# copied out of a log is a claim, one produced here is evidence.
tokenizer = transformers.AutoTokenizer.from_pretrained(T.MODEL_NAME)
cal = json.load(open(os.path.join(OUT, "calibration.json"), encoding="utf-8"))
temperature = float(cal["temperature"])
print(f"temperature from calibration.json: {temperature}")

print("recomputing group-aware val metrics...")
t0 = time.time()
from torch.utils.data import DataLoader, Subset  # noqa: E402

full = T.AdversarialDataset(
    ["datasets/ml-augmented-v7.jsonl"], {n: i for i, n in enumerate(T.ALL_LABELS)}
)
# Same seed/fractions as the training run -> identical split, no leak.
_, _, val_idx = T.group_aware_three_way_split(full.groups, 0.12, 0.08, 42)
val_loader = DataLoader(
    Subset(full, val_idx),
    batch_size=64,
    shuffle=False,
    collate_fn=lambda b: T.collate_fn(b, tokenizer, 256),
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
final = T.evaluate(model, val_loader, device, T.ALL_LABELS, temperature=temperature)
print(
    f"val eval done ({time.time() - t0:.1f}s)  n={len(val_idx)}  "
    f"acc={final['accuracy']:.4f}  atk_rec={final['attack_recall']:.4f}"
)

# ── Re-run the export stage (the part that crashed) ──
model.cpu()
onnx_path = T.export_to_onnx(model, tokenizer, OUT, temperature=temperature)
print(f"[OK] {onnx_path}")

labels_path = os.path.join(OUT, "labels.json")
with open(labels_path, "w", encoding="utf-8") as f:
    json.dump({str(i): n for i, n in enumerate(T.ALL_LABELS)}, f, indent=2)
print(f"[OK] {labels_path}")

T.save_tokenizer(OUT, tokenizer)

report = final["classification_report"]
with open(os.path.join(OUT, "training_stats.json"), "w", encoding="utf-8") as f:
    json.dump(
        {
            "product_name": T.PRODUCT_NAME,
            "product_version": T.PRODUCT_VERSION,
            "base_model": T.MODEL_NAME,
            "num_labels": len(T.ALL_LABELS),
            "labels": T.ALL_LABELS,
            "recovered_export": True,
            "recovery_note": (
                "Training finished; export crashed on missing onnxscript. This run "
                "re-exported from the best checkpoint (pytorch_model.bin) and "
                "recomputed val metrics on the same seed-42 group-aware split. "
                "Per-epoch history was not persisted by the crashed run."
            ),
            "calibration": cal,
            "final_metrics": {
                "accuracy": final["accuracy"],
                "f1_macro": final["f1_macro"],
                "f1_weighted": final["f1_weighted"],
                "attack_recall": final["attack_recall"],
                "attack_precision": final["attack_precision"],
                "split": "group_aware_validation",
            },
            "per_label_metrics": {n: report[n] for n in T.ALL_LABELS if n in report},
        },
        f,
        indent=2,
    )
print("[OK] training_stats.json")

print("\nartifacts in", OUT)
for name in sorted(os.listdir(OUT)):
    p = os.path.join(OUT, name)
    if os.path.isfile(p):
        print(f"  {name:26} {os.path.getsize(p) / 1e6:7.2f} MB")
print("\n-> now run the LAST cell ('5/5') to package and download")

In [ ]:
# 5/5  Verify the artifacts exist, then package and download.
import json, os

d = "models/ml-classifier-v7"
missing = []
for f in ("model.onnx", "labels.json", "calibration.json", "training_stats.json"):
    p = os.path.join(d, f)
    if os.path.exists(p):
        print(f"  OK    {f:22} {os.path.getsize(p)/1e6:6.2f} MB")
    else:
        missing.append(f)
        print(f"  MISS  {f}")
assert not missing, f"training did not produce: {missing} — do not download a partial model"

cal = json.load(open(os.path.join(d, "calibration.json"), encoding="utf-8"))
print("\ncalibration.json (head):")
print(json.dumps(cal, indent=2)[:700])

!zip -r -q ml-classifier-v7.zip models/ml-classifier-v7
print(f"\nzipped: {os.path.getsize('ml-classifier-v7.zip')/1e6:.1f} MB")

from google.colab import files
files.download("ml-classifier-v7.zip")

## After the download

Put `ml-classifier-v7.zip` in the project root. The local side then runs, in this order:

1. **Core-hybrid regression gate** — mitigation recall must stay at or above the
   v4 baseline of **98.15% @ 0.81% FPR**. v6 regressed this and was rolled back;
   if v7 repeats it, v7 does not ship regardless of how good its other numbers look.
2. Cross-distribution recall/FPR on `crossdist-eval-v3.jsonl` (22,681 rows, 0 leak).
3. Per-label + per-language attribution — the aggregate alone points at the wrong fix.
4. Head-to-head vs Meta `Llama-Prompt-Guard-2-86M` on the 16,203 in-scope rows.

All four are wired in `scripts/ml/evaluate-v7.sh`, and step 1 gates steps 2-4.